# Fine-tune ChangeFormer on Sentinel-2 change data (OSCD) and test it on the KEMET1 sites

The LEVIR-CD ChangeFormer scores at chance on the KEMET1 sites because it learned 0.5 m aerial photographs.
This notebook fine-tunes it on **OSCD**, 24 Sentinel-2 image pairs at 10 m with hand-drawn urban-change masks,
then scores it on the 300 KEMET1 sites with the same labels and test sites as every other model.

Two versions are trained: **RGB** (the input ChangeFormer was built for) and **5-band** (adding near-infrared
and SWIR, where vegetation loss and new built-up surfaces show). The one with the better AUC on the 255
development sites is reported; the 45 test sites are only scored, never used for choices.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**. You will be asked
three times to sign in with Google: for Drive (the checkpoint and the results folder), and for Earth Engine.
Total time on a T4 is about 1.5 hours.

In [ ]:
# 1. Settings
EE_PROJECT = "sentinel2-490023"                      # Earth Engine cloud project
CHECKPOINT_FILE_ID = "1RTu5IxtX-XKH5PfgMF2avrH45fWfsvpi"  # best_ckpt.pt (LEVIR-CD) in your Google Drive
ITERATIONS = 2000                                    # fine-tuning steps per version (batch 8, 256 x 256 crops)
RESULTS_FOLDER = "/content/drive/MyDrive/KEMET1_changeformer_oscd"

import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then run again."
DEVICE = "cuda"
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Code: this repository, ChangeFormer, and a few packages
!git clone -q https://github.com/Shahd-M-Elkilany/detecting-encroachment-on-agricultural-land.git kemet1
!cd kemet1 && (git checkout -q two-models 2>/dev/null || true) && git log -1 --format='%h %s'
!git clone -q https://github.com/wgcban/ChangeFormer.git
!pip -q install timm einops rasterio earthengine-api scikit-learn==1.7.2

import sys
sys.path.insert(0, "/content/kemet1/pipeline/changeformer")
sys.path.insert(0, "/content/kemet1")
import finetune_oscd as fo

In [ ]:
# 3. The LEVIR-CD checkpoint from your Drive, and a results folder there
import io, os
from google.colab import auth, drive
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

auth.authenticate_user()
drive.mount("/content/drive")
os.makedirs(RESULTS_FOLDER, exist_ok=True)
if not os.path.exists("best_ckpt.pt"):
    request = build("drive", "v3").files().get_media(fileId=CHECKPOINT_FILE_ID)
    with io.FileIO("best_ckpt.pt", "wb") as fh:
        downloader, done = MediaIoBaseDownload(fh, request, chunksize=64 * 1024 * 1024), False
        while not done:
            _, done = downloader.next_chunk()
print(os.path.getsize("best_ckpt.pt") // 2**20, "MB")

In [ ]:
# 4. OSCD: 14 training and 10 test Sentinel-2 pairs with change masks
fo.download_oscd("/content/oscd")
oscd = fo.load_oscd("/content/oscd")

In [ ]:
# 5. Fine-tune both versions, check them on the OSCD test pairs, save the weights to Drive
models, oscd_scores = {}, {}
for variant in ("rgb", "msi5"):
    channels = fo.CHANNELS[variant]
    net = fo.build_model("/content/ChangeFormer", "best_ckpt.pt", len(channels), DEVICE)
    before = fo.evaluate_oscd(net, oscd["test"], channels, DEVICE) if variant == "rgb" else None
    fo.train(net, oscd["train"], channels, iters=ITERATIONS, batch=8, device=DEVICE)
    oscd_scores[variant] = {"before_finetuning": before, "after": fo.evaluate_oscd(net, oscd["test"], channels, DEVICE)}
    print(variant, oscd_scores[variant])
    torch.save({"model_G_state_dict": net.state_dict(), "channels": channels},
               f"{RESULTS_FOLDER}/changeformer_oscd_{variant}.pt")
    models[variant] = net

In [ ]:
# 6. The 600 KEMET1 Sentinel-2 tiles, rebuilt in Earth Engine on the original grids (about 25 minutes)
import ee
ee.Authenticate()
fo.download_kemet_s2("/content/kemet1/data/tile_grids.json", "/content/s2_tiles", EE_PROJECT)
print(len(os.listdir("/content/s2_tiles")), "tiles")

In [ ]:
# 7. Score all 300 sites with both fine-tuned versions
SCORES = "/content/kemet1/results/changeformer_scores.csv"
for variant, net in models.items():
    fo.score_kemet(net, "/content/s2_tiles", fo.CHANNELS[variant], f"oscd_{variant}", SCORES,
                   "/content/changeformer_oscd_maps", DEVICE)

In [ ]:
# 8. Compare with every other model on the same labels and test sites
import json, shutil, subprocess
out = subprocess.run([sys.executable, "compare_models.py"], cwd="/content/kemet1/pipeline",
                     capture_output=True, text=True)
print(out.stderr[-2000:])
result = json.load(open("/content/kemet1/results/model_comparison.json"))
print(f"{'method':45s} {'dev AUC':>8s} {'test AUC':>9s}   test 95% CI")
for name, m in result["methods"].items():
    print(f"{name:45s} {m['dev_auc']:8.2f} {m['test_auc']:9.2f}   {m['test_auc_ci95']}")
print("chosen fine-tuned version:", result["changeformer_variant_search_dev_auc"].get("oscd_finetuned"))

In [ ]:
# 9. Save everything to Drive (KEMET1_changeformer_oscd) so it can be added to the repository
json.dump(oscd_scores, open(f"{RESULTS_FOLDER}/oscd_test_scores.json", "w"), indent=2)
shutil.copy(SCORES, RESULTS_FOLDER)
shutil.copy("/content/kemet1/results/model_comparison.json", RESULTS_FOLDER)
shutil.make_archive(f"{RESULTS_FOLDER}/changeformer_oscd_maps", "zip", "/content/changeformer_oscd_maps")
print(sorted(os.listdir(RESULTS_FOLDER)))